In [1]:
import os
import pandas as pd
from pathlib import Path
from transformers import T5ForConditionalGeneration, T5Tokenizer


c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR  = Path(os.getcwd()).parent
INPUT_CSV = BASE_DIR / "data" / "processed" / "emails_with_sentiment.csv"

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} emails")
df[['subject', 'body_clean', 'intent', 'sentiment']].head(3)

Loaded 8,064 emails


,subject,body_clean,intent,sentiment
0,Re:,Traveling to have a business meeting takes the...,meeting_request,NEGATIVE
1,NaN,"Randy,\n\n Can you send me a schedule of the s...",meeting_request,NEGATIVE
2,Re: Hello,"Greg,\n\n How about either next Tuesday or Thu...",greeting,NEGATIVE


In [11]:
print(" Loading Flan-T5 model")
model_name = "google/flan-t5-small"
tokenizer  = T5Tokenizer.from_pretrained(model_name)
model      = T5ForConditionalGeneration.from_pretrained(model_name)
print("Flan-T5 model loaded!")

 Loading Flan-T5 model


c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4312.86it/s]
[transformers] The tie

Flan-T5 model loaded!


In [16]:
def generate_reply(email_body, intent, sentiment, style="formal"):
    if sentiment == "NEGATIVE" and intent == "complaint":
        tone = "apologetic and helpful"
    elif sentiment == "POSITIVE" and intent == "appreciation":
        tone = "warm and grateful"
    elif intent == "meeting_request":
        tone = "professional and confirmatory"
    elif intent == "information_request":
        tone = "clear and informative"
    else:
        tone = "professional and polite"

    # Better prompt
    prompt = f"""Write a short professional email reply in {tone} tone.
Original email: {email_body[:200]}
Reply:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    outputs = model.generate(
        inputs["input_ids"],
        max_length=200,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
        temperature=0.7,
    )

    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return reply

In [17]:
print("Testing reply generation on 5 sample emails...\n")
print("=" * 60)

samples = df.sample(5, random_state=42)

for i, row in samples.iterrows():
    print(f"ORIGINAL EMAIL:")
    print(f"   Subject  : {row['subject']}")
    print(f"   Intent   : {row['intent']}")
    print(f"   Sentiment: {row['sentiment']}")
    print(f"   Body     : {row['body_clean'][:200]}")
    print(f"\n GENERATED REPLY:")
    reply = generate_reply(row['body_clean'], row['intent'], row['sentiment'])
    print(f"   {reply}")
    print("=" * 60)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Testing reply generation on 5 sample emails...

ORIGINAL EMAIL:
   Subject  : RE:
   Intent   : meeting_request
   Sentiment: NEGATIVE
   Body     : I have been riding my bike to work a couple of times each week and lifting with Matt. I thought you came in later. 

 
From: Landry, Chad 
Sent: Wednesday, August 22, 2001 11:22 AM
To: Allen, Phillip 

 GENERATED REPLY:
   I have been riding my bike to work a couple of times each week and lifting with Matt. I thought you came in later.
ORIGINAL EMAIL:
   Subject  : astros tix
   Intent   : meeting_request
   Sentiment: POSITIVE
   Body     : Hello:
Wondering if we can meet to pick up the tix. I work and live in downtown. Please advise or call me at 713-557-3330.

 GENERATED REPLY:
   I need to pick up the tix. I work and live in downtown.
ORIGINAL EMAIL:
   Subject  : FYI-Risk Moving
   Intent   : general
   Sentiment: NEGATIVE
   Body     : The Risk group will be moving to the other side of the 32 floor (between the 
kitchen and Bob Hall'

In [18]:

df_sample = df.head(100).copy()
df_sample['generated_reply'] = df_sample.apply(
    lambda row: generate_reply(
        row['body_clean'],
        row['intent'],
        row['sentiment']
    ), axis=1
)
print("Done!")
print(f"Total replies generated: {len(df_sample):,}")

Done!
Total replies generated: 100


In [15]:
OUTPUT_CSV = BASE_DIR / "data" / "processed" / "emails_with_replies.csv"
df.to_csv(OUTPUT_CSV, index=False)
print(f" Saved → {OUTPUT_CSV}")

 Saved → d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_with_replies.csv
